# `Homo_gridblock_Iceland.m` — Python/Jupyter port
Numerically matched port using the shared `gridblock_jupyter.py` implementation. MATLAB's geographic pair definitions, rotated-grid interpolation, accumulators, homogeneity definition, optional bootstrap, and MAT field names are retained.

In [20]:
from pathlib import Path
import numpy as np

from gridblock_jupyter import run_iceland


# ============================================================
# 1. 基本设置
# ============================================================

ini = "_rough"

data_root = Path(
    "/meddy/simingzhang/Data/Parcels_data"
)

subdirs = {
    "_grid": "tranV_onetime_spectukey",
    "_rough": "tranV_onetime_roughdistr_tukey",
    "_rough_1mon": "tranV_onetime_roughdistr_tukey",
    "_rough_2mon": "tranV_onetime_roughdistr_tukey",

    "_roughsmall": "tranV_onetime_roughsmallregion",
    "_roughsmall_1mon": "tranV_onetime_roughsmallregion",
    "_roughsmall_2mon": "tranV_onetime_roughsmallregion",
    "_roughsmall_3mon": "tranV_onetime_roughsmallregion",

    "_roughLASER": "tranV_onetime_roughLASER",
    "_roughsmall_rot": "tranV_onetime_roughsmallregion_rot",
    "_roughsmall_div": "tranV_onetime_roughsmallregion_div",

    "_cruise": "tranV_cruise_roughsmallregion",

    "_roughsmall_500m": "tranV_onetime_roughsmallregion_500m",
    "_rough_500m": "tranV_onetime_rough_500m",

    "_roughbox200g_500m": "tranV_onetime_roughbox200g_500m",
    "_roughbox100g_500m": "tranV_onetime_roughbox100g_500m",
}


# ============================================================
# 2. 以 Eulerian 尺度为标准
# ============================================================

# 单位：km
r_requested_km = np.array([
    2.00000000,
    2.60352118,
    3.38916126,
    4.41187656,
    5.74320702,
    7.47628055,
    9.73232737,
    12.66916020,
    16.49221344,
    21.46891348,
    27.94738544,
    36.38080492,
    47.35909802,
    61.65020731,
    80.25381015,
    104.47124713,
    135.99655214,
    177.03495174,
    230.45712296,
    300.00000000
], dtype=float)


# 检查尺度
r_requested_km = np.sort(
    np.unique(
        r_requested_km[
            np.isfinite(r_requested_km)
            & (r_requested_km > 0)
        ]
    )
)

print("Eulerian reference scales [km]:")
print(r_requested_km)


# ============================================================
# 3. 检查 ini 和输入路径
# ============================================================

if ini not in subdirs:
    raise KeyError(
        f"Unknown ini={ini}. "
        f"Available options are:\n{subdirs.keys()}"
    )

input_dir = data_root / subdirs[ini]

print("\nInitial deployment:", ini)
print("Input directory:", input_dir)


# ============================================================
# 4. 配置 Lagrangian 计算
# ============================================================

cfg = dict(
    case="wave",
    nparticles=289,
    days=89.5,
    dt=3600.0,
    ini=ini,

    timerange_matlab=np.arange(
        1,
        1941
    ),

    nblock_I=4,
    nblock_J=4,

    min_pairs=200,
    min_valid_blocks=8,

    do_bootstrap=True,
    num_boot=1000,
    random_seed=None,

    input_dir=input_dir,

    # 关键参数：
    # 直接使用 Eulerian 的尺度，单位 km
    # 不再使用 xscale
    r_requested_km=r_requested_km,

    grid_file=Path(
        "/meddy/simingzhang/Data/RB_iceland_data/"
        "niskin2km_500m_grd.nc"
    )
)


# ============================================================
# 5. 运行
# ============================================================

print("\nStarting Lagrangian calculation...")
print("Number of scales:", len(cfg["r_requested_km"]))
print("Scales [km]:", cfg["r_requested_km"])

result, output_path = run_iceland(cfg)

print("\nCalculation finished.")
print("Output file:")
print(output_path)

Eulerian reference scales [km]:
[  2.           2.60352118   3.38916126   4.41187656   5.74320702
   7.47628055   9.73232737  12.6691602   16.49221344  21.46891348
  27.94738544  36.38080492  47.35909802  61.65020731  80.25381015
 104.47124713 135.99655214 177.03495174 230.45712296 300.        ]

Initial deployment: _rough
Input directory: /meddy/simingzhang/Data/Parcels_data/tranV_onetime_roughdistr_tukey

Starting Lagrangian calculation...
Number of scales: 20
Scales [km]: [  2.           2.60352118   3.38916126   4.41187656   5.74320702
   7.47628055   9.73232737  12.6691602   16.49221344  21.46891348
  27.94738544  36.38080492  47.35909802  61.65020731  80.25381015
 104.47124713 135.99655214 177.03495174 230.45712296 300.        ]


Iceland Lagrangian pair statistics: 100%|██████████████████| 1939/1939 [02:21<00:00, 13.69time/s]


saved /meddy/simingzhang/Data/Parcels_data/tranV_onetime_roughdistr_tukey/wave_pars_P289T1940_roughgridBlockHL.mat

Calculation finished.
Output file:
/meddy/simingzhang/Data/Parcels_data/tranV_onetime_roughdistr_tukey/wave_pars_P289T1940_roughgridBlockHL.mat


## Reproducibility note
The MATLAB script does not seed its Iceland bootstrap. Leave `random_seed=None` for equivalent unseeded behavior, or set an integer for repeatable Python runs. Exact cross-language random draws require supplying identical saved permutations/resampling indices to both programs; deterministic pre-bootstrap fields can be compared directly.

# H3 Eul+Lag:EXP-F,EXP-R

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from matplotlib.lines import Line2D


# ============================================================
# 1. 路径
# ============================================================

root = Path("/meddy/simingzhang/Data")
figure_dir = root / "Figure_SF3"
parcels_dir = root / "Parcels_data"

files = {
    ("HIT", "Eulerian"):
        figure_dir / "HIT_Eulerian_gridBlock_all_orders.mat",

    ("HF", "Eulerian"):
        figure_dir / "HF_Eulerian_gridBlock_all_orders.mat",

    ("SM", "Eulerian"):
        figure_dir / "SM_Eulerian_gridBlock_all_orders.mat",

    ("HF", "EXP-R"):
        parcels_dir / "tranV_onetime_roughdistr_tukey" /
        "wave_pars_P289T1940_roughgridBlockHL.mat",

    ("SM", "EXP-R"):
        parcels_dir / "tranV_onetime_roughdistr_tukey" /
        "nowave_pars_P289T1940_roughgridBlockHL.mat",

    ("HF", "EXP-F"):
        parcels_dir / "tranV_onetime_roughsmallregion" /
        "wave_pars_P289T1940_roughsmallgridBlockHL.mat",

    ("SM", "EXP-F"):
        parcels_dir / "tranV_onetime_roughsmallregion" /
        "nowave_pars_P289T1940_roughsmallgridBlockHL.mat",
}


# 自动寻找 HIT Lagrangian 文件
hit_dir = parcels_dir / "HIT2d_rough"

hit_candidates = sorted(
    list(hit_dir.glob("*EulerianScalesHL.mat"))
    + list(hit_dir.glob("*gridBlockHL.mat")),
    key=lambda p: p.stat().st_mtime
)

if len(hit_candidates) == 0:
    raise FileNotFoundError(
        f"No HIT Lagrangian file found in:\n{hit_dir}"
    )

files[("HIT", "Lagrangian")] = hit_candidates[-1]


# ============================================================
# 2. 工具函数
# ============================================================

def find_variable(data, names, description):

    for name in names:
        if name in data:
            return np.asarray(
                data[name],
                dtype=float
            ).squeeze()

    available = [
        key for key in data.keys()
        if not key.startswith("__")
    ]

    raise KeyError(
        f"{description} not found.\n"
        f"Tried: {names}\n"
        f"Available: {available}"
    )


def reduce_to_scale(value, nscale):

    value = np.asarray(
        value,
        dtype=float
    ).squeeze()

    if value.ndim == 1:
        return value

    if value.ndim == 2:

        if value.shape[-1] == nscale:
            return np.nanmean(value, axis=0)

        if value.shape[0] == nscale:
            return np.nanmean(value, axis=1)

    raise ValueError(
        f"Cannot reduce shape {value.shape} "
        f"to {nscale} scales."
    )


def clean_data(r, *values):

    r = np.ravel(
        np.asarray(r, dtype=float)
    )

    values = [
        np.ravel(
            np.asarray(value, dtype=float)
        )
        for value in values
    ]

    n = min(
        [len(r)] + [len(value) for value in values]
    )

    r = r[:n]
    values = [
        value[:n]
        for value in values
    ]

    valid = (
        np.isfinite(r)
        & (r > 0)
    )

    for value in values:
        valid &= np.isfinite(value)

    r = r[valid]
    values = [
        value[valid]
        for value in values
    ]

    order = np.argsort(r)

    return r[order], [
        value[order]
        for value in values
    ]


# ============================================================
# 3. 直接从原始 MAT 文件读取
# ============================================================

def load_one(filename, case_name, method_name):

    if not filename.exists():
        raise FileNotFoundError(
            f"File not found:\n{filename}"
        )

    data = loadmat(
        filename,
        squeeze_me=True
    )

    print(
        f"\nReading {case_name} / {method_name}:"
    )
    print(filename)

    # --------------------------------------------------------
    # separation scale
    # --------------------------------------------------------

    if method_name == "Eulerian":

        r = find_variable(
            data,
            [
                "r_requested",
                "r_actual",
                "dist_axis",
                "r"
            ],
            "Eulerian scale"
        )

        r = np.ravel(r)

        # Iceland Eulerian scale: m -> km
        if case_name in ["HF", "SM"]:
            r = r / 1000.0

    else:

        r = find_variable(
            data,
            [
                "r_requested_km",
                "r_requested",
                "dist_axis",
                "r"
            ],
            "Lagrangian scale"
        )

        r = np.ravel(r)

    # --------------------------------------------------------
    # H3L
    # --------------------------------------------------------

    H3 = find_variable(
        data,
        [
            "H3L",
            "H3",
            "H_Dlll",
            "H_L3",
            "H_L_3"
        ],
        "H3L"
    )

    H3 = reduce_to_scale(
        H3,
        len(r)
    )

    # --------------------------------------------------------
    # mean and RMS
    # --------------------------------------------------------

    if method_name == "Eulerian":

        mean_value = find_variable(
            data,
            [
                "mean_Dlll",
                "mean_D3",
                "mean_SF3"
            ],
            "Eulerian mean Dlll"
        )

        rms_value = find_variable(
            data,
            [
                "rms_Dlll",
                "rms_D3",
                "rms_SF3"
            ],
            "Eulerian RMS Dlll"
        )

    else:

        mean_value = find_variable(
            data,
            [
                "mean_SF3",
                "mean_Dlll",
                "mean_D3"
            ],
            "Lagrangian mean SF3"
        )

        rms_value = find_variable(
            data,
            [
                "rms_SF3",
                "rms_Dlll",
                "rms_D3"
            ],
            "Lagrangian RMS SF3"
        )

    mean_value = reduce_to_scale(
        mean_value,
        len(r)
    )

    rms_value = reduce_to_scale(
        rms_value,
        len(r)
    )

    # --------------------------------------------------------
    # cancellation ratio
    # --------------------------------------------------------

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):

        cancellation = (
            np.abs(mean_value)
            / rms_value
        )

    r, values = clean_data(
        r,
        H3,
        cancellation
    )

    H3, cancellation = values

    print(
        f"n scales = {len(r)}"
    )
    print(
        f"r[:5] = {r[:5]}"
    )
    print(
        f"H3[:5] = {H3[:5]}"
    )
    print(
        f"cancellation[:5] = {cancellation[:5]}"
    )

    return {
        "r": r,
        "H3": H3,
        "cancellation": cancellation
    }


# ============================================================
# 4. 读取所有数据
# ============================================================

data_all = {}

for key, filename in files.items():

    data_all[key] = load_one(
        filename,
        key[0],
        key[1]
    )


# ============================================================
# 5. 绘图风格
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "stix",

    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,

    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8
})


case_list = [
    ("HIT", "2D turbulence"),
    ("HF", "HF"),
    ("SM", "SM")
]

case_colors = {
    "HIT": "black",
    "HF": "#0072B2",
    "SM": "#D55E00"
}

method_styles = {
    "Eulerian": "-",
    "Lagrangian": "--",
    "EXP-R": "--",
    "EXP-F": "-."
}


# ============================================================
# 6. 创建图
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(8.0, 5.6),
    sharex="col",
    sharey="row"
)


# ============================================================
# 7. 第一行：H3
# ============================================================

for col, (case_name, title) in enumerate(case_list):

    ax = axes[0, col]

    for (current_case, method_name), item in data_all.items():

        if current_case != case_name:
            continue

        r = item["r"]
        H3 = item["H3"]

        ax.semilogx(
            r,
            H3,
            color=case_colors[case_name],
            linestyle=method_styles.get(
                method_name,
                "--"
            ),
            linewidth=1.7
        )

    ax.set_title(
        title,
        pad=6,
        fontsize=10
    )

    ax.text(
        0.04,
        0.94,
        f"({chr(97 + col)})",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        fontweight="bold"
    )

    ax.set_ylim(
        0,
        1.0
    )

    ax.set_yticks(
        np.arange(
            0,
            1.01,
            0.25
        )
    )

    if col == 0:
        ax.set_ylabel(
            r"$H_L^3(r)$"
        )
    else:
        ax.tick_params(
            axis="y",
            labelleft=False
        )

    ax.grid(
        True,
        which="major",
        color="0.85",
        linewidth=0.6
    )

    ax.grid(
        False,
        which="minor"
    )


# ============================================================
# 8. 第二行：cancellation ratio
# ============================================================

for col, (case_name, title) in enumerate(case_list):

    ax = axes[1, col]

    for (current_case, method_name), item in data_all.items():

        if current_case != case_name:
            continue

        r = item["r"]
        cancellation = item["cancellation"]

        ax.semilogx(
            r,
            cancellation,
            color=case_colors[case_name],
            linestyle=method_styles.get(
                method_name,
                "--"
            ),
            linewidth=1.7
        )

    ax.text(
        0.04,
        0.94,
        f"({chr(100 + col)})",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        fontweight="bold"
    )

    ax.set_ylim(
        0,
        1.0
    )

    ax.set_yticks(
        np.arange(
            0,
            1.01,
            0.25
        )
    )

    if col == 0:
        ax.set_ylabel(
            r"$|\mathrm{mean}(D_{LLL})|/"
            r"\mathrm{rms}(D_{LLL})$"
        )
    else:
        ax.tick_params(
            axis="y",
            labelleft=False
        )

    ax.set_xlabel(
        r"$r$"
        if case_name == "HIT"
        else r"$r\ [\mathrm{km}]$"
    )

    ax.grid(
        True,
        which="major",
        color="0.85",
        linewidth=0.6
    )

    ax.grid(
        False,
        which="minor"
    )


# ============================================================
# 9. 总标题
# ============================================================

fig.suptitle(
    "Scale-dependent homogeneity diagnostics",
    fontsize=12,
    fontweight="bold",
    y=0.985
)


# ============================================================
# 10. 无边框 legend
# ============================================================

legend_handles = [
    Line2D(
        [],
        [],
        color="black",
        linestyle="-",
        linewidth=1.7,
        label="Eulerian"
    ),

    Line2D(
        [],
        [],
        color="black",
        linestyle="--",
        linewidth=1.7,
        label="Lagrangian EXP-R"
    ),

    Line2D(
        [],
        [],
        color="black",
        linestyle="-.",
        linewidth=1.7,
        label="Lagrangian EXP-F"
    )
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.055),
    ncol=3,
    frameon=False,
    fontsize=9,
    handlelength=2.3,
    handletextpad=0.5,
    columnspacing=1.2
)


# ============================================================
# 11. 布局与保存
# ============================================================

fig.subplots_adjust(
    left=0.13,
    right=0.985,
    bottom=0.22,
    top=0.88,
    wspace=0.12,
    hspace=0.18
)

output_file = (
    figure_dir
    / "Fig_H3_cancellation_two_rows_JPO.png"
)

fig.savefig(
    output_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("\nSaved:")
print(output_file)


Reading HIT / Eulerian:
/meddy/simingzhang/Data/Figure_SF3/HIT_Eulerian_gridBlock_all_orders.mat
n scales = 25
r[:5] = [0.02454369 0.03004279 0.03677399 0.04501333 0.05509873]
H3[:5] = [0.01235028 0.01026878 0.00965984 0.00813477 0.00813477]
cancellation[:5] = [0.99969499 0.99978913 0.99981339 0.99986766 0.99986766]

Reading HF / Eulerian:
/meddy/simingzhang/Data/Figure_SF3/HF_Eulerian_gridBlock_all_orders.mat
n scales = 20
r[:5] = [2.         2.60352118 3.38916126 4.41187656 5.74320702]
H3[:5] = [0.18961446 0.18961446 0.19801121 0.22367829 0.23496753]
cancellation[:5] = [0.93058831 0.93058831 0.92454172 0.9047039  0.89535779]

Reading SM / Eulerian:
/meddy/simingzhang/Data/Figure_SF3/SM_Eulerian_gridBlock_all_orders.mat
n scales = 20
r[:5] = [2.         2.60352118 3.38916126 4.41187656 5.74320702]
H3[:5] = [0.16895478 0.16895478 0.1845468  0.23518522 0.26080787]
cancellation[:5] = [0.94449305 0.94449305 0.93412838 0.89517397 0.87262281]

Reading HF / EXP-R:
/meddy/simingzhang/Data/Pa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# 检查变量
# ============================================================

if "results" not in globals():
    raise NameError(
        "results is not defined. "
        "Please run the data-loading code first."
    )

if "figure_dir" not in globals():
    figure_dir = "/meddy/simingzhang/Data/Figure_SF3"


# ============================================================
# JPO / article style
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "stix",

    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,

    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8
})


# ============================================================
# Case 和方法设置
# ============================================================

case_list = [
    ("HIT", "2D turbulence", "(a)"),
    ("HF", "HF", "(b)"),
    ("SM", "SM", "(c)")
]

case_colors = {
    "HIT": "black",
    "HF": "#0072B2",
    "SM": "#D55E00"
}

method_styles = {
    "Eulerian": "-",
    "Lagrangian": "--",
    "EXP-R": "--",
    "EXP-F": "-."
}


# ============================================================
# 创建图
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(8.0, 3.6),
    sharey=True
)


# ============================================================
# 绘制 H3
# ============================================================

for col, (case_name, title, panel_label) in enumerate(
    case_list
):

    ax = axes[col]

    for (current_case, method_name), item in results.items():

        if current_case != case_name:
            continue

        # HIT 只有 Eulerian 和 Lagrangian
        if (
            case_name == "HIT"
            and method_name not in [
                "Eulerian",
                "Lagrangian"
            ]
        ):
            continue

        r = np.asarray(
            item["r"],
            dtype=float
        ).squeeze()

        H3 = np.asarray(
            item["H3"],
            dtype=float
        ).squeeze()

        n = min(
            len(r),
            len(H3)
        )

        r = r[:n]
        H3 = H3[:n]

        valid = (
            np.isfinite(r)
            & np.isfinite(H3)
            & (r > 0)
        )

        if not np.any(valid):
            continue

        ax.semilogx(
            r[valid],
            H3[valid],
            color=case_colors[case_name],
            linestyle=method_styles.get(
                method_name,
                "--"
            ),
            linewidth=1.7
        )

    # --------------------------------------------------------
    # 标题
    # --------------------------------------------------------

    ax.set_title(
        title,
        pad=6
    )

    # --------------------------------------------------------
    # panel label
    # --------------------------------------------------------

    ax.text(
        0.04,
        0.94,
        panel_label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        fontweight="bold"
    )

    # --------------------------------------------------------
    # y 轴
    # --------------------------------------------------------

    ax.set_ylim(
        0,
        1.0
    )

    ax.set_yticks(
        np.arange(
            0,
            1.01,
            0.25
        )
    )

    if col == 0:
        ax.set_ylabel(
            r"$H_L^3(r)$"
        )
    else:
        ax.tick_params(
            axis="y",
            which="both",
            labelleft=False
        )

    # --------------------------------------------------------
    # x 轴
    # --------------------------------------------------------

    ax.set_xlabel(
        r"$r$"
        if case_name == "HIT"
        else r"$r\ [\mathrm{km}]$"
    )

    # --------------------------------------------------------
    # 网格
    # --------------------------------------------------------

    ax.grid(
        True,
        which="major",
        color="0.85",
        linewidth=0.6
    )

    ax.grid(
        False,
        which="minor"
    )

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)


# ============================================================
# 总标题
# ============================================================

fig.suptitle(
    "Scale-dependent homogeneity diagnostics",
    fontsize=12,
    fontweight="bold",
    y=0.985
)


# ============================================================
# 无边框图例
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="-",
        linewidth=1.7,
        label="Eulerian"
    ),

    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=1.7,
        label="Lagrangian EXP-R"
    ),

    Line2D(
        [0],
        [0],
        color="black",
        linestyle="-.",
        linewidth=1.7,
        label="Lagrangian EXP-F"
    )
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.005),
    ncol=3,
    frameon=False,
    fontsize=9,
    handlelength=2.4,
    handletextpad=0.6,
    columnspacing=1.4
)


# ============================================================
# 布局和保存
# ============================================================

fig.subplots_adjust(
    left=0.105,
    right=0.985,
    bottom=0.25,
    top=0.87,
    wspace=0.10
)

output_file = (
    figure_dir
    / "Fig_H3_HIT_HF_SM_Eulerian_Lagrangian_only_toprow_polished.png"
)

fig.savefig(
    output_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:")
print(output_file)

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from matplotlib.lines import Line2D


# ============================================================
# 1. 路径
# ============================================================

root = Path("/meddy/simingzhang/Data")
figdir = root / "Figure_SF3"
pdir = root / "Parcels_data"

files = {
    ("HIT", "Eulerian"):
        figdir / "HIT_Eulerian_gridBlock_all_orders.mat",

    ("HIT", "Lagrangian"):
        pdir / "HIT2d_rough"
        / "HIT2d_pars_P30000T150_EulerianScalesHL.mat",

    ("HF", "Eulerian"):
        figdir / "HF_Eulerian_gridBlock_all_orders.mat",

    ("HF", "EXP-R"):
        pdir / "tranV_onetime_roughdistr_tukey"
        / "wave_pars_P289T1940_roughgridBlockHL.mat",

    ("HF", "EXP-F"):
        pdir / "tranV_onetime_roughsmallregion"
        / "wave_pars_P289T1940_roughsmallgridBlockHL.mat",

    ("SM", "Eulerian"):
        figdir / "SM_Eulerian_gridBlock_all_orders.mat",

    ("SM", "EXP-R"):
        pdir / "tranV_onetime_roughdistr_tukey"
        / "nowave_pars_P289T1940_roughgridBlockHL.mat",

    ("SM", "EXP-F"):
        pdir / "tranV_onetime_roughsmallregion"
        / "nowave_pars_P289T1940_roughsmallgridBlockHL.mat",
}


# ============================================================
# 2. 读取数据的函数
# ============================================================

def find_variable(data, names, label):

    for name in names:
        if name in data:
            return np.asarray(
                data[name],
                dtype=float
            ).squeeze()

    available = [
        key for key in data.keys()
        if not key.startswith("__")
    ]

    raise KeyError(
        f"{label} not found.\n"
        f"Tried: {names}\n"
        f"Available variables: {available}"
    )


def reduce_to_scale(value, nscale):

    value = np.asarray(
        value,
        dtype=float
    ).squeeze()

    if value.ndim == 1:
        return value

    if value.ndim == 2:

        if value.shape[-1] == nscale:
            return np.nanmean(value, axis=0)

        if value.shape[0] == nscale:
            return np.nanmean(value, axis=1)

    raise ValueError(
        f"Cannot match array shape {value.shape} "
        f"to {nscale} scales."
    )


def load_H2_H3(filename, case_name, method_name):

    if not filename.exists():
        raise FileNotFoundError(
            f"File not found:\n{filename}"
        )

    data = loadmat(
        filename,
        squeeze_me=True
    )

    # --------------------------------------------------------
    # separation scale
    # --------------------------------------------------------

    if method_name == "Eulerian":

        r = find_variable(
            data,
            [
                "r_requested",
                "r_actual",
                "dist_axis",
                "r"
            ],
            "Eulerian scale"
        )

        r = np.ravel(r)

        # Iceland Eulerian distances are stored in meters
        if case_name in ["HF", "SM"]:
            r = r / 1000.0

    else:

        r = find_variable(
            data,
            [
                "r_requested_km",
                "r_requested",
                "dist_axis",
                "r"
            ],
            "Lagrangian scale"
        )

        r = np.ravel(r)

    # --------------------------------------------------------
    # H2L and H3L
    # --------------------------------------------------------

    H2 = find_variable(
        data,
        [
            "H2L",
            "H2",
            "HL2",
            "H_L2",
            "H_L_2"
        ],
        "H2L"
    )

    H3 = find_variable(
        data,
        [
            "H3L",
            "H3",
            "HL3",
            "H_L3",
            "H_L_3"
        ],
        "H3L"
    )

    H2 = reduce_to_scale(
        H2,
        len(r)
    )

    H3 = reduce_to_scale(
        H3,
        len(r)
    )

    n = min(
        len(r),
        len(H2),
        len(H3)
    )

    r = r[:n]
    H2 = H2[:n]
    H3 = H3[:n]

    valid = (
        np.isfinite(r)
        & np.isfinite(H2)
        & np.isfinite(H3)
        & (r > 0)
    )

    r = r[valid]
    H2 = H2[valid]
    H3 = H3[valid]

    order = np.argsort(r)

    return {
        "r": r[order],
        "H2": H2[order],
        "H3": H3[order]
    }


# ============================================================
# 3. 载入全部数据
# ============================================================

results = {}

for key, filename in files.items():

    results[key] = load_H2_H3(
        filename,
        key[0],
        key[1]
    )

    print(
        f"Loaded {key[0]} / {key[1]}: "
        f"{filename.name}"
    )


# ============================================================
# 4. 文章统一风格
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "stix",

    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,

    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
})


case_colors = {
    "HIT": "black",
    "HF": "#0072B2",
    "SM": "#D55E00"
}

method_styles = {
    "Eulerian": "-",
    "Lagrangian": "--",
    "EXP-R": "--",
    "EXP-F": "-."
}

cases = [
    ("HIT", "2D turbulence"),
    ("HF", "HF"),
    ("SM", "SM")
]

orders = [
    (2, r"$H_L^2(r)$"),
    (3, r"$H_L^3(r)$")
]


# ============================================================
# 5. 创建图
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(8.0, 5.6),
    sharex="col",
    sharey="row"
)

fig.subplots_adjust(
    left=0.13,
    right=0.94,
    bottom=0.18,
    top=0.90,
    wspace=0.16,
    hspace=0.24
)


# ============================================================
# 6. 绘制 H2/H3
# ============================================================

panel_labels = [
    "(a)", "(b)", "(c)",
    "(d)", "(e)", "(f)"
]

panel_index = 0

for row, (order, row_label) in enumerate(orders):

    for col, (case_name, title) in enumerate(cases):

        ax = axes[row, col]

        # ----------------------------------------------------
        # 绘制三种方法
        # ----------------------------------------------------

        for (current_case, method_name), item in results.items():

            if current_case != case_name:
                continue

            # HIT 只有 Eulerian 和 Lagrangian
            if (
                case_name == "HIT"
                and method_name not in [
                    "Eulerian",
                    "Lagrangian"
                ]
            ):
                continue

            linestyle = method_styles.get(
                method_name,
                "--"
            )

            ax.semilogx(
                item["r"],
                item[f"H{order}"],
                color=case_colors[case_name],
                linestyle=linestyle,
                linewidth=1.8
            )

        # ----------------------------------------------------
        # column titles
        # ----------------------------------------------------

        if row == 0:
            ax.set_title(
                title,
                fontsize=10,
                pad=6
            )

        # ----------------------------------------------------
        # panel label
        # ----------------------------------------------------

        ax.text(
            0.04,
            0.94,
            panel_labels[panel_index],
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=10,
            fontweight="bold"
        )

        panel_index += 1

        # ----------------------------------------------------
        # axes limits and ticks
        # ----------------------------------------------------

        ax.set_ylim(
            0,
            1.0
        )

        ax.set_yticks(
            np.arange(
                0,
                1.01,
                0.25
            )
        )

        # 每行只在第一列显示 y label
        if col == 0:
            ax.set_ylabel(
                row_label,
                fontsize=11
            )
        else:
            ax.tick_params(
                axis="y",
                which="both",
                labelleft=False
            )

        # 只在底行显示 x label
        if row == 1:

            ax.set_xlabel(
                r"$r$"
                if case_name == "HIT"
                else r"$r\ [\mathrm{km}]$",
                fontsize=11
            )

        else:

            ax.tick_params(
                axis="x",
                which="both",
                labelbottom=False
            )

        # ----------------------------------------------------
        # grid
        # ----------------------------------------------------

        ax.grid(
            True,
            which="major",
            color="0.85",
            linewidth=0.6
        )

        ax.grid(
            False,
            which="minor"
        )


# ============================================================
# 7. 行标签
# ============================================================

pos_top = axes[0, 0].get_position()
pos_bottom = axes[1, 0].get_position()

y_top = 0.5 * (
    pos_top.y0 + pos_top.y1
)

y_bottom = 0.5 * (
    pos_bottom.y0 + pos_bottom.y1
)



# ============================================================
# 8. 无边框 legend
# ============================================================

legend_handles = [
    Line2D(
        [],
        [],
        color="black",
        linestyle="-",
        linewidth=1.8,
        label="Eulerian"
    ),

    Line2D(
        [],
        [],
        color="black",
        linestyle="--",
        linewidth=1.8,
        label="Lagrangian EXP-R"
    ),

    Line2D(
        [],
        [],
        color="black",
        linestyle="-.",
        linewidth=1.8,
        label="Lagrangian EXP-F"
    )
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.075),
    ncol=3,
    frameon=False,
    handlelength=2.2,
    columnspacing=1.2,
    handletextpad=0.5,
    fontsize=9
)


# ============================================================
# 9. 保存
# ============================================================

output_file = (
    figdir
    / "Fig_H2_H3_HIT_HF_SM_Eulerian_Lagrangian_JPO.png"
)

fig.savefig(
    output_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:")
print(output_file)